A — Membangun Katalog Menu

In [1]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")

db_kopi = client["kopi_nusantara"]
koleksi_menu = db_kopi["menu"]

koleksi_menu.delete_many({})

menu = [
    {
        "nama_menu": "Espresso",
        "harga": 18000,
        "kategori": "Kopi",
        "tingkat_kepahitan": 5,
        "stok": 30
    },
    {
        "nama_menu": "Americano",
        "harga": 20000,
        "kategori": "Kopi",
        "tingkat_kepahitan": 4,
        "stok": 35
    },
    {
        "nama_menu": "Cappuccino",
        "harga": 25000,
        "kategori": "Kopi",
        "tingkat_kepahitan": 3,
        "stok": 25
    },
    {
        "nama_menu": "Cafe Latte",
        "harga": 27000,
        "kategori": "Kopi",
        "tingkat_kepahitan": 2,
        "stok": 25
    },
    {
        "nama_menu": "Mocha",
        "harga": 30000,
        "kategori": "Kopi",
        "tingkat_kepahitan": 3,
        "stok": 20
    },
    {
        "nama_menu": "Cold Brew",
        "harga": 24000,
        "kategori": "Kopi",
        "tingkat_kepahitan": 4,
        "stok": 20
    },
    {
        "nama_menu": "Roti Bakar",
        "harga": 18000,
        "kategori": "Makanan",
        "pedas_level": 0,
        "stok": 25
    },
    {
        "nama_menu": "Mie Goreng",
        "harga": 22000,
        "kategori": "Makanan",
        "pedas_level": 2,
        "stok": 20
    },
    {
        "nama_menu": "Kentang Goreng",
        "harga": 20000,
        "kategori": "Makanan",
        "pedas_level": 0,
        "stok": 30
    },
    {
        "nama_menu": "Nasi Ayam",
        "harga": 30000,
        "kategori": "Makanan",
        "pedas_level": 1,
        "stok": 20
    },
    {
        "nama_menu": "Pisang Goreng",
        "harga": 15000,
        "kategori": "Makanan",
        "pedas_level": 0,
        "stok": 25
    },
    {
        "nama_menu": "Roti Maryam",
        "harga": 17000,
        "kategori": "Makanan",
        "pedas_level": 0,
        "stok": 15
    }
]

koleksi_menu.insert_many(menu)

print(
    "Jumlah menu:",
    koleksi_menu.count_documents({})
)

Jumlah menu: 12


B — Kueri Bertingkat

In [26]:
print("Seluruh menu kategori Kopi dengan tingkat_kepahitan di atas 3 :")
for menu in koleksi_menu.find(
    {
        "kategori": "Kopi",
        "tingkat_kepahitan": {"$gt": 3}
    },
    {
        "_id": 0,
        "nama_menu": 1,
        "harga": 1,
        "tingkat_kepahitan": 1
    }
):
    print(
        menu["nama_menu"],
        menu["harga"],
        menu["tingkat_kepahitan"]
    )
print("\n")

print("Seluruh menu dengan harga di antara 15000 dan 30000 :")
for m in koleksi_menu.find({
    "harga": {
        "$gte": 15000,
        "$lte": 30000
    }
}):
    print(m["nama_menu"], m["harga"])
print("\n")

print("Menu yang tidak memiliki field pedas_level sama sekali :")
for m in koleksi_menu.find({
    "pedas_level": {"$exists": False}
}):
    print(m["nama_menu"], m["kategori"])

Seluruh menu kategori Kopi dengan tingkat_kepahitan di atas 3 :
Espresso 18000 5
Americano 20000 4
Cold Brew 24000 4


Seluruh menu dengan harga di antara 15000 dan 30000 :
Espresso 18000
Americano 20000
Cappuccino 25000
Cafe Latte 27000
Mocha 30000
Cold Brew 24000
Roti Bakar 18000
Mie Goreng 22000
Kentang Goreng 20000
Nasi Ayam 30000
Pisang Goreng 15000
Roti Maryam 17000


Menu yang tidak memiliki field pedas_level sama sekali :
Espresso Kopi
Americano Kopi
Cappuccino Kopi
Cafe Latte Kopi
Mocha Kopi
Cold Brew Kopi


C - Memperbarui Data

*Tambahkan field status bernilai "Tersedia" ke seluruh dokumen menggunakan update_many(). Kemudian, ubah status menjadi "Habis" khusus untuk 2 menu pilihan anda menggunakan update_one().*

In [28]:
koleksi_menu.update_many(
    {},
    {"$set": {"status": "Tersedia"}}
)
        
koleksi_menu.update_one(
    {"nama_menu": "Espresso"},
    {"$set": {"status": "Habis"}}
)

koleksi_menu.update_one(
    {"nama_menu": "Roti Bakar"},
    {"$set": {"status": "Habis"}}
)

for m in koleksi_menu.find(
    {},
    {
        "_id": 0,
        "nama_menu": 1,
        "status": 1
    }
):
    print(m)

{'nama_menu': 'Espresso', 'status': 'Habis'}
{'nama_menu': 'Americano', 'status': 'Tersedia'}
{'nama_menu': 'Cappuccino', 'status': 'Tersedia'}
{'nama_menu': 'Cafe Latte', 'status': 'Tersedia'}
{'nama_menu': 'Mocha', 'status': 'Tersedia'}
{'nama_menu': 'Cold Brew', 'status': 'Tersedia'}
{'nama_menu': 'Roti Bakar', 'status': 'Habis'}
{'nama_menu': 'Mie Goreng', 'status': 'Tersedia'}
{'nama_menu': 'Kentang Goreng', 'status': 'Tersedia'}
{'nama_menu': 'Nasi Ayam', 'status': 'Tersedia'}
{'nama_menu': 'Pisang Goreng', 'status': 'Tersedia'}
{'nama_menu': 'Roti Maryam', 'status': 'Tersedia'}


D - Integrasi MongoDB dengan PySpark

In [36]:
from pyspark.sql import SparkSession
import pandas as pd

spark = SparkSession.builder \
    .appName("Tugas9-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# Mengambil seluruh data dari collection menu
dokumen_menu = list(koleksi_menu.find({}))

# Mengubah menjadi Pandas DataFrame
df_pandas = pd.DataFrame(dokumen_menu)

# Menyesuaikan tipe data MongoDB
df_pandas["_id"] = df_pandas["_id"].astype(str)

# Menghapus kolom _id jika tidak diperlukan
df_pandas = df_pandas.drop(columns=["_id"])

# Mengisi field kosong dengan None
df_pandas = df_pandas.astype(object).where(
    pd.notna(df_pandas), None
)

# Konversi ke Spark DataFrame
df_spark_menu = spark.createDataFrame(df_pandas)

# 1. Menampilkan struktur DataFrame
df_spark_menu.printSchema()

# Menampilkan seluruh data
df_spark_menu.show(truncate=False)

# 2. Menghitung rata-rata harga per kategori
from pyspark.sql.functions import avg

df_spark_menu.groupBy("kategori").agg(
    avg("harga").alias("rata_rata_harga")
).show()

root
 |-- nama_menu: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- tingkat_kepahitan: double (nullable = true)
 |-- stok: long (nullable = true)
 |-- status: string (nullable = true)
 |-- pedas_level: double (nullable = true)

+--------------+-----+--------+-----------------+----+--------+-----------+
|nama_menu     |harga|kategori|tingkat_kepahitan|stok|status  |pedas_level|
+--------------+-----+--------+-----------------+----+--------+-----------+
|Espresso      |18000|Kopi    |5.0              |30  |Habis   |NULL       |
|Americano     |20000|Kopi    |4.0              |35  |Tersedia|NULL       |
|Cappuccino    |25000|Kopi    |3.0              |25  |Tersedia|NULL       |
|Cafe Latte    |27000|Kopi    |2.0              |25  |Tersedia|NULL       |
|Mocha         |30000|Kopi    |3.0              |20  |Tersedia|NULL       |
|Cold Brew     |24000|Kopi    |4.0              |20  |Tersedia|NULL       |
|Roti Bakar    |18000|Makanan 

 **Kesimpulan**

 *Tantangan yang saya temui saat mengonversi data MongoDB menjadi Spark DataFrame adalah struktur dokumen yang tidak selalu seragam. Pada MongoDB, setiap dokumen dapat memiliki field yang berbeda sesuai dengan jenis datanya. Contohnya, menu dengan kategori Kopi memiliki field tingkat_kepahitan, sedangkan menu dengan kategori Makanan tidak memiliki field tersebut. Hal ini menjadi tantangan karena Spark DataFrame membutuhkan struktur kolom dan tipe data yang lebih seragam agar data dapat diproses dengan baik.*

*Untuk mengatasi masalah tersebut, saya menggunakan Pandas DataFrame sebagai perantara sebelum data dikonversi ke Spark DataFrame. Selain itu, saya melakukan penyesuaian tipe data pada kolom _id menjadi string dan menangani nilai yang kosong dengan mengubahnya menjadi None atau null. Dengan cara ini, dokumen yang memiliki field berbeda tetap dapat digabungkan ke dalam satu DataFrame tanpa harus menghilangkan data yang tersedia.*